# Improved Melanoma Detection Model

## Improvements over previous experiments:
1. **Data Augmentation** - Rotation, flipping, zoom, brightness
2. **Two-Stage Training** - Head training + fine-tuning
3. **Fixed Loss Function** - Correct `from_logits=False` with sigmoid
4. **Medical Metrics** - AUC, Sensitivity, Specificity
5. **Modern Architecture** - EfficientNet-B4 (or Xception option)
6. **Class Weights** - Handle imbalanced data
7. **Early Stopping** - Prevent overfitting
8. **Focal Loss Option** - Better for hard examples

## 1. Environment Setup

In [ ]:
!nvidia-smi -q -i 0 | grep "Product Name"

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

In [ ]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f'Your runtime has {ram_gb:.1f} gigabytes of available RAM')
if ram_gb < 20:
    print('Warning: Not using a high-RAM runtime')
else:
    print('You are using a high-RAM runtime!')

In [ ]:
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Detect hardware
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
    strategy = tf.distribute.TPUStrategy(tpu)
    print("Running on TPU")
except ValueError:
    strategy = tf.distribute.MirroredStrategy()
    print("Running on GPU/CPU")

print(f"Number of accelerators: {strategy.num_replicas_in_sync}")

## 2. Configuration

In [ ]:
import os
import numpy as np
from matplotlib import pyplot as plt

# ============================================================
# CONFIGURATION - Adjust these parameters
# ============================================================

# Model selection
MODEL_NAME = 'EfficientNetB4'  # Options: 'EfficientNetB4', 'Xception', 'EfficientNetV2S'

# Input sizes for different models
MODEL_INPUT_SIZES = {
    'EfficientNetB4': (380, 380),
    'Xception': (299, 299),
    'EfficientNetV2S': (384, 384),
}
INPUT_SHAPE = MODEL_INPUT_SIZES[MODEL_NAME]

# Training parameters
EPOCHS_STAGE1 = 20      # Head training only
EPOCHS_STAGE2 = 30      # Fine-tuning
BATCH_SIZE = 16 if strategy.num_replicas_in_sync == 1 else 16 * strategy.num_replicas_in_sync

# Learning rates
LR_STAGE1 = 1e-3        # Higher LR for head training
LR_STAGE2 = 1e-5        # Lower LR for fine-tuning

# Fine-tuning: number of layers to freeze (from bottom)
FREEZE_LAYERS = {
    'EfficientNetB4': 200,
    'Xception': 100,
    'EfficientNetV2S': 150,
}

# Data split
VALIDATION_SPLIT = 0.15
TEST_SPLIT = 0.15

# Class weights (adjust if melanoma is underrepresented)
USE_CLASS_WEIGHTS = True
CLASS_WEIGHTS = {0: 1.2, 1: 1.0}  # Melanoma = 0, NotMelanoma = 1

# Use Focal Loss instead of Binary Crossentropy
USE_FOCAL_LOSS = False

print(f"Model: {MODEL_NAME}")
print(f"Input shape: {INPUT_SHAPE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Stage 1 epochs: {EPOCHS_STAGE1}, Stage 2 epochs: {EPOCHS_STAGE2}")

## 3. Data Preparation

In [ ]:
# Download dataset from Kaggle
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download yauhenbichel/melanoma
!unzip -q melanoma.zip

In [ ]:
# Check dataset
melanoma_count = len(os.listdir('melanoma/Melanoma'))
not_melanoma_count = len(os.listdir('melanoma/NotMelanoma'))
total_images = melanoma_count + not_melanoma_count

print(f"Melanoma images: {melanoma_count}")
print(f"Not Melanoma images: {not_melanoma_count}")
print(f"Total images: {total_images}")
print(f"Class balance: {melanoma_count/total_images:.2%} / {not_melanoma_count/total_images:.2%}")

In [ ]:
from shutil import copyfile
import random

# Create directories
for split in ['training', 'validation', 'testing']:
    for cls in ['Melanoma', 'NotMelanoma']:
        os.makedirs(f'{split}/{cls}', exist_ok=True)

def split_data(source, training, validation, testing, val_split, test_split):
    """Split data into train/validation/test sets."""
    files = [f for f in os.listdir(source) if os.path.getsize(os.path.join(source, f)) > 0]
    random.shuffle(files)
    
    n = len(files)
    n_test = int(n * test_split)
    n_val = int(n * val_split)
    n_train = n - n_test - n_val
    
    for i, f in enumerate(files):
        src = os.path.join(source, f)
        if i < n_train:
            dst = os.path.join(training, f)
        elif i < n_train + n_val:
            dst = os.path.join(validation, f)
        else:
            dst = os.path.join(testing, f)
        copyfile(src, dst)
    
    return n_train, n_val, n_test

# Split melanoma images
m_train, m_val, m_test = split_data(
    'melanoma/Melanoma',
    'training/Melanoma', 'validation/Melanoma', 'testing/Melanoma',
    VALIDATION_SPLIT, TEST_SPLIT
)

# Split not melanoma images
nm_train, nm_val, nm_test = split_data(
    'melanoma/NotMelanoma',
    'training/NotMelanoma', 'validation/NotMelanoma', 'testing/NotMelanoma',
    VALIDATION_SPLIT, TEST_SPLIT
)

print(f"\nData Split:")
print(f"Training:   {m_train + nm_train} images (Melanoma: {m_train}, NotMelanoma: {nm_train})")
print(f"Validation: {m_val + nm_val} images (Melanoma: {m_val}, NotMelanoma: {nm_val})")
print(f"Testing:    {m_test + nm_test} images (Melanoma: {m_test}, NotMelanoma: {nm_test})")

## 4. Data Augmentation (NEW!)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ============================================================
# ENHANCED DATA AUGMENTATION
# Skin lesions are rotation/flip invariant - use aggressive augmentation
# ============================================================

train_datagen = ImageDataGenerator(
    rescale=1./255.,
    rotation_range=360,           # Full rotation (lesions have no orientation)
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,           # Skin lesions are orientation invariant
    fill_mode='reflect',          # Better than 'nearest' for medical images
    brightness_range=[0.7, 1.3],  # Handle lighting variations
)

# NO augmentation for validation/test - only rescaling
validation_datagen = ImageDataGenerator(rescale=1./255.)
test_datagen = ImageDataGenerator(rescale=1./255.)

# Create generators
train_generator = train_datagen.flow_from_directory(
    'training/',
    target_size=INPUT_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

validation_generator = validation_datagen.flow_from_directory(
    'validation/',
    target_size=INPUT_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    'testing/',
    target_size=INPUT_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"\nClass mapping: {train_generator.class_indices}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")

In [ ]:
# Visualize augmented samples
def show_augmented_samples(generator, n_samples=9):
    """Display augmented training samples."""
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    for i, ax in enumerate(axes.flat):
        img, label = next(generator)
        ax.imshow(img[0])
        ax.set_title(f"Label: {'Melanoma' if label[0] == 0 else 'Not Melanoma'}")
        ax.axis('off')
    plt.suptitle('Augmented Training Samples', fontsize=14)
    plt.tight_layout()
    plt.show()

show_augmented_samples(train_generator)

## 5. Model Architecture

In [ ]:
from tensorflow.keras import layers, regularizers, Model
from tensorflow.keras.applications import EfficientNetB4, Xception, EfficientNetV2S

def create_base_model(model_name, input_shape):
    """Create base model (pretrained on ImageNet)."""
    base_models = {
        'EfficientNetB4': EfficientNetB4,
        'Xception': Xception,
        'EfficientNetV2S': EfficientNetV2S,
    }
    
    base = base_models[model_name](
        weights='imagenet',
        include_top=False,
        input_shape=(*input_shape, 3)
    )
    return base

def create_model(model_name, input_shape, trainable_base=False):
    """
    Create complete model with improved classification head.
    
    Improvements:
    - BatchNormalization for stable training
    - Larger dense layer (256 vs 64)
    - Multiple dropout layers
    """
    base = create_base_model(model_name, input_shape)
    base.trainable = trainable_base
    
    # Improved classification head
    inputs = layers.Input(shape=(*input_shape, 3))
    x = base(inputs, training=False)  # Important: training=False for BatchNorm
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)  # Sigmoid for binary classification
    
    model = Model(inputs, outputs)
    return model, base

# Create model (base frozen initially)
with strategy.scope():
    model, base_model = create_model(MODEL_NAME, INPUT_SHAPE, trainable_base=False)

model.summary()

In [ ]:
# Count trainable parameters
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])

print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Non-trainable parameters: {non_trainable_params:,}")
print(f"Total parameters: {trainable_params + non_trainable_params:,}")

## 6. Metrics & Loss Function (FIXED!)

In [ ]:
from tensorflow.keras.metrics import AUC, Precision, Recall, BinaryAccuracy
from tensorflow.keras.losses import BinaryCrossentropy

# ============================================================
# MEDICAL METRICS - Critical for melanoma detection
# ============================================================

def get_metrics():
    """Get metrics important for medical diagnosis."""
    return [
        BinaryAccuracy(name='accuracy'),
        AUC(name='auc'),
        Recall(name='sensitivity'),      # True Positive Rate - CRITICAL
        Precision(name='precision'),
        # Specificity at 90% sensitivity
        tf.keras.metrics.SpecificityAtSensitivity(0.9, name='spec_at_90_sens'),
        # Sensitivity at 90% specificity  
        tf.keras.metrics.SensitivityAtSpecificity(0.9, name='sens_at_90_spec'),
    ]

# ============================================================
# LOSS FUNCTION OPTIONS
# ============================================================

def get_loss():
    """Get loss function."""
    if USE_FOCAL_LOSS:
        # Focal Loss - better for hard examples and class imbalance
        # Requires: pip install tensorflow-addons
        try:
            import tensorflow_addons as tfa
            return tfa.losses.SigmoidFocalCrossEntropy(alpha=0.25, gamma=2.0)
        except ImportError:
            print("tensorflow-addons not installed, using BinaryCrossentropy")
            return BinaryCrossentropy(from_logits=False)
    else:
        # Standard Binary Crossentropy
        # FIXED: from_logits=False because we use sigmoid activation
        return BinaryCrossentropy(from_logits=False)

print(f"Loss function: {'Focal Loss' if USE_FOCAL_LOSS else 'Binary Crossentropy'}")
print(f"Using class weights: {USE_CLASS_WEIGHTS}")
if USE_CLASS_WEIGHTS:
    print(f"Class weights: {CLASS_WEIGHTS}")

## 7. Callbacks

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, 
    LearningRateScheduler, TensorBoard
)
import datetime

def get_callbacks(stage, monitor='val_auc'):
    """Get callbacks for training."""
    
    callbacks = [
        # Early stopping - monitor AUC (more important than accuracy for medical)
        EarlyStopping(
            monitor=monitor,
            patience=10,
            mode='max',
            restore_best_weights=True,
            verbose=1
        ),
        
        # Save best model
        ModelCheckpoint(
            f'best_model_{MODEL_NAME}_stage{stage}.h5',
            monitor=monitor,
            save_best_only=True,
            mode='max',
            verbose=1
        ),
        
        # Reduce LR on plateau
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),
    ]
    
    return callbacks

print("Callbacks configured:")
print("  - EarlyStopping (patience=10, monitor=val_auc)")
print("  - ModelCheckpoint (save best model)")
print("  - ReduceLROnPlateau (factor=0.5, patience=5)")

## 8. Stage 1: Train Classification Head Only

In [ ]:
from tensorflow.keras.optimizers import Adam

print("=" * 60)
print("STAGE 1: Training Classification Head Only")
print("=" * 60)
print(f"Base model frozen: {not base_model.trainable}")
print(f"Learning rate: {LR_STAGE1}")
print(f"Epochs: {EPOCHS_STAGE1}")

# Compile model for Stage 1
with strategy.scope():
    model.compile(
        optimizer=Adam(learning_rate=LR_STAGE1),
        loss=get_loss(),
        metrics=get_metrics()
    )

In [ ]:
import time

start_time = time.time()

# Train Stage 1
history_stage1 = model.fit(
    train_generator,
    epochs=EPOCHS_STAGE1,
    validation_data=validation_generator,
    callbacks=get_callbacks(stage=1),
    class_weight=CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None,
    verbose=1
)

stage1_time = time.time() - start_time
print(f"\nStage 1 completed in {stage1_time/60:.1f} minutes")

In [ ]:
# Plot Stage 1 results
def plot_training_history(history, title="Training History"):
    """Plot training metrics."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Accuracy
    axes[0, 0].plot(history.history['accuracy'], label='Train')
    axes[0, 0].plot(history.history['val_accuracy'], label='Validation')
    axes[0, 0].set_title('Accuracy')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Loss
    axes[0, 1].plot(history.history['loss'], label='Train')
    axes[0, 1].plot(history.history['val_loss'], label='Validation')
    axes[0, 1].set_title('Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # AUC
    axes[1, 0].plot(history.history['auc'], label='Train')
    axes[1, 0].plot(history.history['val_auc'], label='Validation')
    axes[1, 0].set_title('AUC (Area Under ROC Curve)')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Sensitivity (Recall)
    axes[1, 1].plot(history.history['sensitivity'], label='Train Sensitivity')
    axes[1, 1].plot(history.history['val_sensitivity'], label='Val Sensitivity')
    axes[1, 1].set_title('Sensitivity (Recall)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_training_history(history_stage1, "Stage 1: Head Training Only")

## 9. Stage 2: Fine-Tuning (NEW!)

In [ ]:
print("=" * 60)
print("STAGE 2: Fine-Tuning Base Model")
print("=" * 60)

# Unfreeze base model for fine-tuning
base_model.trainable = True

# Freeze early layers (feature extraction layers)
freeze_until = FREEZE_LAYERS[MODEL_NAME]
for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

# Count trainable layers
trainable_layers = sum(1 for layer in base_model.layers if layer.trainable)
frozen_layers = len(base_model.layers) - trainable_layers

print(f"Base model layers: {len(base_model.layers)}")
print(f"Frozen layers: {frozen_layers}")
print(f"Trainable layers: {trainable_layers}")
print(f"Learning rate: {LR_STAGE2} (10x lower than Stage 1)")
print(f"Epochs: {EPOCHS_STAGE2}")

In [ ]:
# Recompile with lower learning rate for fine-tuning
with strategy.scope():
    model.compile(
        optimizer=Adam(learning_rate=LR_STAGE2),
        loss=get_loss(),
        metrics=get_metrics()
    )

# Count parameters after unfreezing
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print(f"\nTrainable parameters after unfreezing: {trainable_params:,}")

In [ ]:
start_time = time.time()

# Train Stage 2 (Fine-tuning)
history_stage2 = model.fit(
    train_generator,
    epochs=EPOCHS_STAGE2,
    validation_data=validation_generator,
    callbacks=get_callbacks(stage=2),
    class_weight=CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None,
    verbose=1
)

stage2_time = time.time() - start_time
print(f"\nStage 2 completed in {stage2_time/60:.1f} minutes")

In [ ]:
plot_training_history(history_stage2, "Stage 2: Fine-Tuning")

## 10. Evaluation on Test Set

In [ ]:
# Load best model
best_model = tf.keras.models.load_model(f'best_model_{MODEL_NAME}_stage2.h5')

# Evaluate on test set
print("=" * 60)
print("FINAL EVALUATION ON TEST SET")
print("=" * 60)

test_generator.reset()
results = best_model.evaluate(test_generator, verbose=1)

print("\nTest Results:")
for metric, value in zip(best_model.metrics_names, results):
    print(f"  {metric}: {value:.4f}")

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    roc_curve, auc, precision_recall_curve
)
import seaborn as sns

# Get predictions
test_generator.reset()
y_pred_proba = best_model.predict(test_generator)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
y_true = test_generator.classes

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Melanoma', 'NotMelanoma']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Melanoma', 'NotMelanoma'],
            yticklabels=['Melanoma', 'NotMelanoma'])
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['Melanoma', 'NotMelanoma'],
            yticklabels=['Melanoma', 'NotMelanoma'])
axes[1].set_title('Confusion Matrix (Normalized)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

# Calculate key metrics
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)  # Recall for melanoma
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)  # Positive Predictive Value
npv = tn / (tn + fn)  # Negative Predictive Value

print(f"\nKey Medical Metrics:")
print(f"  Sensitivity (Recall): {sensitivity:.4f} - Ability to detect melanoma")
print(f"  Specificity: {specificity:.4f} - Ability to rule out melanoma")
print(f"  PPV (Precision): {ppv:.4f} - Probability that positive prediction is correct")
print(f"  NPV: {npv:.4f} - Probability that negative prediction is correct")

In [ ]:
# ROC Curve and Precision-Recall Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(y_true, y_pred_proba)
roc_auc = auc(fpr, tpr)

axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.3)
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].grid(True)

# Precision-Recall Curve
precision_vals, recall_vals, thresholds_pr = precision_recall_curve(y_true, y_pred_proba)
pr_auc = auc(recall_vals, precision_vals)

axes[1].plot(recall_vals, precision_vals, 'g-', linewidth=2, label=f'PR (AUC = {pr_auc:.4f})')
axes[1].fill_between(recall_vals, precision_vals, alpha=0.3, color='green')
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='lower left')
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\nROC AUC: {roc_auc:.4f}")
print(f"PR AUC: {pr_auc:.4f}")

## 11. Save Final Model

In [ ]:
# Save in multiple formats
import json

# Save Keras format (.h5)
model_name_base = f'melanoma_{MODEL_NAME}_improved'
best_model.save(f'{model_name_base}.h5')
print(f"Saved: {model_name_base}.h5")

# Save TensorFlow SavedModel format (for TF Serving / Lambda)
best_model.save(f'{model_name_base}_savedmodel', save_format='tf')
print(f"Saved: {model_name_base}_savedmodel/")

# Save metadata
metadata = {
    'model_name': MODEL_NAME,
    'input_shape': list(INPUT_SHAPE) + [3],
    'class_mapping': {'Melanoma': 0, 'NotMelanoma': 1},
    'test_accuracy': float(results[1]),
    'test_auc': float(results[2]),
    'test_sensitivity': float(sensitivity),
    'test_specificity': float(specificity),
    'training_config': {
        'epochs_stage1': EPOCHS_STAGE1,
        'epochs_stage2': EPOCHS_STAGE2,
        'batch_size': BATCH_SIZE,
        'lr_stage1': LR_STAGE1,
        'lr_stage2': LR_STAGE2,
        'data_augmentation': True,
        'fine_tuning': True,
        'class_weights': CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None,
    }
}

with open(f'{model_name_base}_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Saved: {model_name_base}_metadata.json")

## 12. Summary

In [ ]:
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"\nModel: {MODEL_NAME}")
print(f"Input shape: {INPUT_SHAPE}")
print(f"\nTraining Configuration:")
print(f"  Stage 1 (Head only): {EPOCHS_STAGE1} epochs, LR={LR_STAGE1}")
print(f"  Stage 2 (Fine-tuning): {EPOCHS_STAGE2} epochs, LR={LR_STAGE2}")
print(f"  Data augmentation: Yes")
print(f"  Class weights: {CLASS_WEIGHTS if USE_CLASS_WEIGHTS else 'No'}")
print(f"\nTest Set Results:")
print(f"  Accuracy: {results[1]:.4f}")
print(f"  AUC: {roc_auc:.4f}")
print(f"  Sensitivity: {sensitivity:.4f}")
print(f"  Specificity: {specificity:.4f}")
print(f"\nTotal training time: {(stage1_time + stage2_time)/60:.1f} minutes")
print(f"\nSaved models:")
print(f"  - {model_name_base}.h5")
print(f"  - {model_name_base}_savedmodel/")
print(f"  - {model_name_base}_metadata.json")